## Initialization

In [ ]:
from vespa.application import Vespa
import json
import pandas as pd
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from local .env file
import ipynbname
nb_path = ipynbname.path()
notebook_dir = nb_path.parent
env_path = notebook_dir / '.env'
load_dotenv(dotenv_path=env_path)

VESPA_ENDPOINT = os.getenv('VESPA_ENDPOINT')
VESPA_CERT_PATH = os.getenv('VESPA_CERT_PATH')
VESPA_KEY_PATH = os.getenv('VESPA_KEY_PATH')

vespa_app = Vespa(url=VESPA_ENDPOINT, cert=VESPA_CERT_PATH, key=VESPA_KEY_PATH)
print("App object initialized")

# Visualizing price and rating quantiles

## Quantiles with pyvespa

In [ ]:
with vespa_app.syncio(connections=1) as session:
    response = session.query(yql='''
    select * from product where true limit 0 | 
      all( group('all') each(
        output(quantiles([0.1,0.3,0.5,0.7,0.9],Price))
        output(quantiles([0.1,0.3,0.5,0.7,0.9],AverageRating))
    ))
    ''')
print(json.dumps(response.get_json(), indent=2))

## Load results in a dataframe

In [ ]:
grouping_result = response.get_json()['root']['children'][0]['children'][0]['children'][0]['fields']
price_data = grouping_result["quantiles([0.1,0.3,0.5,0.7,0.9],Price)"]
df = pd.DataFrame(price_data)
print(df)

## Visualize pricing quantiles

In [ ]:
import plotly.graph_objects as go

def plot_quantiles(df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df['quantile'],
        y=df['value'],
        mode='lines+markers',
        name='value',
        line=dict(color='rgb(255, 20, 147)', width=2),
        marker=dict(color='rgb(255, 20, 147)', size=8)
    ))

    fig.update_layout(
        xaxis_title='quantile',
        hovermode='x unified',
        template='plotly_white',
        width=800,
        height=500
    )

    fig.show()

plot_quantiles(df)

## Same thing for average rating

In [ ]:
rating_data = grouping_result["quantiles([0.1,0.3,0.5,0.7,0.9],AverageRating)"]
df = pd.DataFrame(rating_data)
plot_quantiles(df)

# Visualizing relevance judgements

In [ ]:
df = pd.read_csv(
    "../../ch2/evaluation/judgements.csv",
    dtype={
        "query_id": "int16",
        "document_id": "string",
        "rating": "int8",
    },
)

quantiles = [0.1, 0.3, 0.5, 0.7, 0.9]

# compute per-query quantiles
judgements_df = df.groupby("query_id")["rating"].quantile(quantiles, interpolation="nearest").unstack()
judgements_df.columns = [str(q) for q in quantiles]
print(judgements_df)

In [ ]:
# Sample 50 queries and plot median rating per query
sampled_query_ids = df['query_id'].unique()
sampled_query_ids = pd.Series(sampled_query_ids).sample(n=50, random_state=42).sort_values().tolist()

# Compute median rating for each sampled query
median_ratings = []
for query_id in sampled_query_ids:
    query_ratings = df[df['query_id'] == query_id]['rating']
    median_ratings.append({
        'query_id': query_id,
        'median_rating': query_ratings.median()
    })

median_df = pd.DataFrame(median_ratings)

# Plot median ratings
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=median_df['query_id'],
    y=median_df['median_rating'],
    mode='lines+markers',
    name='median_rating',
    line=dict(color='rgb(255, 20, 147)', width=2),
    marker=dict(color='rgb(255, 20, 147)', size=8)
))

fig.update_layout(
    xaxis_title='query_id',
    yaxis_title='median_rating',
    hovermode='x unified',
    template='plotly_white',
    width=800,
    height=500
)

fig.show()


# Visualizing query scores

## First, run the queries to get the document scores and features
e.g., vector and lexical scores

In [ ]:
# read queries CSV
df = pd.read_csv(
    "../../ch2/evaluation/queries.csv",
    dtype={
        "query_id": "int16",
        "query_text": "string"
    },
)

# query body template
query_body = {
  "yql": '''
    select * from product where ({targetHits:100}nearestNeighbor(ProductName_embedding,q_embedding)) OR ({targetHits:100}nearestNeighbor(Description_embedding,q_embedding)) OR
      userQuery()
    ''',
  "query": "QUERY TEXT WILL GO HERE",
  "input.query(q_embedding)": "embed(@query)",
  "ranking.profile": "hybrid"
}

# build all query requests
queries = []
for query in df['query_text'].to_list():
  query_body['query'] = query
  queries.append(query_body)

# run all queries async with pyvespa
responses = await vespa_app.query_many_async(queries=queries)

# collect all responses
all_responses = [response.json for response in responses]

# sample 20 responses
sample_responses = all_responses[:20]

print(json.dumps(sample_responses, indent=2))

## Now extract scores and features from responses

In [ ]:
FEATURES = [
    "closeness_description",
    "closeness_productname",
    "native_rank_description",
    "native_rank_name",
]

rows = []
i = 0
for resp in sample_responses:
    qtext = queries[i]["query"]
    for hit in resp["root"].get("children", []):  # all returned docs for this query
        fields = hit.get("fields", {})
        sf = fields.get("summaryfeatures", {}) or {}
        rows.append({
            "query_index": i,
            "query": qtext,
            "document_id": fields.get("documentid"),
            # (optional) keep some extra identifiers:
            "product_id": fields.get("ProductID"),
            "product_name": fields.get("ProductName"),
            # summary features -> columns
            **{k: sf.get(k) for k in FEATURES},
        })
    i += 1

queries_docs_df = pd.DataFrame(rows)
print(queries_docs_df)

## Plot scores. Notice how semantic scores dominate

In [ ]:
# Plot scatter chart with different colors for each feature
import plotly.graph_objects as go

fig = go.Figure()

# Define colors for each feature
colors = {
    'closeness_description': 'rgb(255, 20, 147)',  # pink
    'closeness_productname': 'rgb(0, 100, 200)',   # blue
    'native_rank_description': 'rgb(0, 200, 100)', # green
    'native_rank_name': 'rgb(255, 165, 0)'        # orange
}

# Add a trace for each feature
# Use numeric index for x-axis to avoid downsampling issues with categorical data
for feature in ["closeness_description", "closeness_productname", "native_rank_description", "native_rank_name"]:
    fig.add_trace(go.Scatter(
        x=list(range(len(queries_docs_df))),  # Use numeric index
        y=queries_docs_df[feature],
        mode='markers',
        name=feature,
        marker=dict(
            color=colors[feature],
            size=8,
            opacity=0.7
        ),
        # Include document_id in hover text
        customdata=queries_docs_df[['document_id']],
        hovertemplate='<b>%{fullData.name}</b><br>' +
                      'Document ID: %{customdata[0]}<br>' +
                      'Value: %{y}<br>' +
                      'Index: %{x}<extra></extra>'
    ))

fig.update_layout(
    xaxis_title='document_id',
    yaxis_title='feature value',
    xaxis=dict(showticklabels=False), # don't show the document ids on the x-axis - too long
    hovermode='closest',
    template='plotly_white',
    width=1200,
    height=600,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ),
    # Ensure all points are shown
    uirevision='constant'
)

# Show all points - disable downsampling
fig.show(config={
    'displayModeBar': True,
    'plotGlPixelRatio': 1,
    'toImageButtonOptions': {
        'format': 'png',
        'filename': 'scatter_plot',
        'height': 600,
        'width': 1200,
        'scale': 1
    }
})
